In [1]:
"""
Task 12: Feature Engineering
Edutech Solution - Data Science Internship
 
Dataset : Housing Dataset (California Housing Prices - housing.csv)
Goal    : Improve model performance through data manipulation
          - Create new interaction features
          - Apply One-Hot and Label Encoding
          - Handle skewed data with log transformation
"""
 
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
 

In [5]:
# 1. Load the dataset
df = pd.read_csv("housing.csv")
 
print("Original shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nMissing values:\n", df.isnull().sum())
 
# Handle missing values (total_bedrooms has 207 nulls) -> fill with median
df["total_bedrooms"] = df["total_bedrooms"].fillna(df["total_bedrooms"].median())
 

Original shape: (20640, 10)

First 5 rows:
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  

Missing values:
 longitude     

In [6]:
# 2. Create new interaction features
df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
df["population_per_household"] = df["population"] / df["households"]
df["income_x_age"] = df["median_income"] * df["housing_median_age"]
 
print("\nNew interaction features (sample):")
print(df[["rooms_per_household", "bedrooms_per_room",
          "population_per_household", "income_x_age"]].head())


New interaction features (sample):
   rooms_per_household  bedrooms_per_room  population_per_household  \
0             6.984127           0.146591                  2.555556   
1             6.238137           0.155797                  2.109842   
2             8.288136           0.129516                  2.802260   
3             5.817352           0.184458                  2.547945   
4             6.281853           0.172096                  2.181467   

   income_x_age  
0      341.3332  
1      174.3294  
2      377.3848  
3      293.4412  
4      200.0024  


In [7]:
# 3. Encoding categorical feature: ocean_proximity
print("\nUnique categories in 'ocean_proximity':", df["ocean_proximity"].unique())
 
# --- Label Encoding ---
label_encoder = LabelEncoder()
df["ocean_proximity_label"] = label_encoder.fit_transform(df["ocean_proximity"])
 
print("\nLabel Encoding mapping:")
for category, code in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f"  {category} -> {code}")
 
# --- One-Hot Encoding ---
onehot_encoder = OneHotEncoder(sparse_output=False)
onehot_array = onehot_encoder.fit_transform(df[["ocean_proximity"]])
onehot_cols = onehot_encoder.get_feature_names_out(["ocean_proximity"])
onehot_df = pd.DataFrame(onehot_array, columns=onehot_cols, index=df.index)
 
df = pd.concat([df, onehot_df], axis=1)
print("\nOne-Hot Encoded columns added:", list(onehot_cols))


Unique categories in 'ocean_proximity': <ArrowStringArray>
['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND']
Length: 5, dtype: str

Label Encoding mapping:
  <1H OCEAN -> 0
  INLAND -> 1
  ISLAND -> 2
  NEAR BAY -> 3
  NEAR OCEAN -> 4

One-Hot Encoded columns added: ['ocean_proximity_<1H OCEAN', 'ocean_proximity_INLAND', 'ocean_proximity_ISLAND', 'ocean_proximity_NEAR BAY', 'ocean_proximity_NEAR OCEAN']


In [8]:
# 4. Handle skewed data with log transformation
skew_check_cols = ["total_rooms", "total_bedrooms", "population",
                    "households", "median_house_value",
                    "population_per_household"]
 
print("\nSkewness BEFORE log transformation:")
print(df[skew_check_cols].skew())
 
log_cols = ["total_rooms", "total_bedrooms", "population",
             "households", "median_house_value", "population_per_household"]
 
for col in log_cols:
    df[f"{col}_log"] = np.log1p(df[col])
 
print("\nSkewness AFTER log1p transformation:")
print(df[[f"{c}_log" for c in log_cols]].skew())


Skewness BEFORE log transformation:
total_rooms                  4.147343
total_bedrooms               3.481141
population                   4.935858
households                   3.410438
median_house_value           0.977763
population_per_household    97.639561
dtype: float64

Skewness AFTER log1p transformation:
total_rooms_log                -1.075533
total_bedrooms_log             -0.998768
population_log                 -1.044087
households_log                 -1.051607
median_house_value_log         -0.173166
population_per_household_log    3.879679
dtype: float64


In [10]:
 #5. Save the enhanced dataset
df.to_csv("EnhancedDataset.csv", index=False)
 
print("\nFinal enhanced dataset shape:", df.shape)
print("Saved as EnhancedDataset.csv")
print("\nFinal columns:")
print(list(df.columns))


Final enhanced dataset shape: (20640, 26)
Saved as EnhancedDataset.csv

Final columns:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household', 'income_x_age', 'ocean_proximity_label', 'ocean_proximity_<1H OCEAN', 'ocean_proximity_INLAND', 'ocean_proximity_ISLAND', 'ocean_proximity_NEAR BAY', 'ocean_proximity_NEAR OCEAN', 'total_rooms_log', 'total_bedrooms_log', 'population_log', 'households_log', 'median_house_value_log', 'population_per_household_log']
